In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, roc_auc_score, confusion_matrix
)

In [4]:
# Preprocessing
df = pd.read_csv('Telco-Customer-Churn.csv')

# Removing it as it is just an identifier
X = df.drop(['customerID', 'Churn'], axis=1)

y = df['Churn'].map({'No':0, 'Yes':1}) # Important to convert to 0-1

# Converting total charges to numeric
X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')
X['TotalCharges'] = X['TotalCharges'].fillna(X['TotalCharges'].median())

# Encoding the categorical features
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [12]:
# Tuned base models

rf_model = RandomForestClassifier(
    max_depth=6, n_estimators=300, min_samples_split=2, 
    min_samples_leaf=1, class_weight='balanced', random_state=42
)

neg, pos = np.bincount(y_train)

scale_pos_weight = neg / pos

xgb_model = XGBClassifier(
    eval_metric = 'logloss', 
    scale_pos_weight = scale_pos_weight,
    random_state=42,
    # Giving Best Parameters from previous notebook using GridSearchCV
    colsample_bytree= 0.8, 
    learning_rate= 0.01, 
    max_depth= 3, 
    n_estimators= 200, 
    subsample= 0.8
)

In [13]:
# Voting Classifier
# (soft voting is used as it uses probabilities and works better with imbalance)

voting_clf = VotingClassifier(
    estimators=[('rf', rf_model), ('xgb', xgb_model)],
    voting = 'soft'
)

voting_clf.fit(X_train, y_train)

y_pred_vote = voting_clf.predict(X_test)
y_proba_vote = voting_clf.predict_proba(X_test)[:, 1]

print('Voting Classifier Results:')
print(f'Accuracy: {accuracy_score(y_test,y_pred_vote):.4f}')
print(f'Recall: {recall_score(y_test,y_pred_vote):.4f}')
print(f'Precision: {precision_score(y_test,y_pred_vote):.4f}')
print(f'AUC: {roc_auc_score(y_test,y_proba_vote):.4f}')

Voting Classifier Results:
Accuracy: 0.7410
Recall: 0.8155
Precision: 0.5075
AUC: 0.8437


In [14]:
# Stacking Classifier
# (meta-model learns how to combine RF + XGBoost)

stacking_clf = StackingClassifier(
    estimators=[('rf', rf_model), ('xgb', xgb_model)],
    final_estimator= LogisticRegression(class_weight='balanced'),
    cv = 5
)

stacking_clf.fit(X_train, y_train)

y_pred_stack = stacking_clf.predict(X_test)
y_proba_stack = stacking_clf.predict_proba(X_test)[:, 1]

print('Stacking Classifier Results:')
print(f'Accuracy: {accuracy_score(y_test,y_pred_stack):.4f}')
print(f'Recall: {recall_score(y_test,y_pred_stack):.4f}')
print(f'Precision: {precision_score(y_test,y_pred_stack):.4f}')
print(f'AUC: {roc_auc_score(y_test,y_proba_stack):.4f}')

Stacking Classifier Results:
Accuracy: 0.7459
Recall: 0.8102
Precision: 0.5136
AUC: 0.8437


In [15]:
# Saving the models

joblib.dump(voting_clf, 'churn_voting_model.pkl')
joblib.dump(stacking_clf, 'churn_stacking_model.pkl')

['churn_stacking_model.pkl']